# Creación de otros componentes que forman parte del modelo

La gran parte de los componentes que forman parte de la arquitectura de la red neuronal artificial pueden sustituirse por componentes personalizados de la misma forma

In [25]:
import tensorflow as tf

tf.__version__

'2.13.0'

In [26]:
from tensorflow.keras import datasets
boston_housing = datasets.boston_housing
(X_train, y_train), (X_test, y_test) = boston_housing.load_data()

In [27]:
import pandas as pd
import numpy as np

features = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]

df_train = pd.DataFrame(np.column_stack([X_train, y_train]), columns=features)
df_train.head(10)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,1.23247,0.0,8.14,0.0,0.538,6.142,91.7,3.9769,4.0,307.0,21.0,396.90,18.72,15.2
1,0.02177,82.5,2.03,0.0,0.415,7.610,15.7,6.2700,2.0,348.0,14.7,395.38,3.11,42.3
2,4.89822,0.0,18.10,0.0,0.631,4.970,100.0,1.3325,24.0,666.0,20.2,375.52,3.26,50.0
3,0.03961,0.0,5.19,0.0,0.515,6.037,34.5,5.9853,5.0,224.0,20.2,396.90,8.01,21.1
4,3.69311,0.0,18.10,0.0,0.713,6.376,88.4,2.5671,24.0,666.0,20.2,391.43,14.65,17.7
5,0.28392,0.0,7.38,0.0,0.493,5.708,74.3,4.7211,5.0,287.0,19.6,391.13,11.74,18.5
6,9.18702,0.0,18.10,0.0,0.700,5.536,100.0,1.5804,24.0,666.0,20.2,396.90,23.60,11.3
7,4.09740,0.0,19.58,0.0,0.871,5.468,100.0,1.4118,5.0,403.0,14.7,396.90,26.42,15.6
8,2.15505,0.0,19.58,0.0,0.871,5.628,100.0,1.5166,5.0,403.0,14.7,169.27,16.65,15.6
9,1.62864,0.0,21.89,0.0,0.624,5.019,100.0,1.4394,4.0,437.0,21.2,396.90,34.41,14.4


In [50]:
# Función de activación personalizada
@tf.keras.utils.register_keras_serializable()
def my_softplus(z):
    return tf.math.log(tf.exp(z) + 1.0)

# Función de inicialización personalizada
@tf.keras.utils.register_keras_serializable()
def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2. / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)

# Función de regularización personalizada
@tf.keras.utils.register_keras_serializable()
def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))

# Restricción personalizada aplicada al valor de los parámetros de una capa
@tf.keras.utils.register_keras_serializable()
def my_positive_weights(weights):
    return tf.where(weights < 0., tf.zeros_like(weights), weights)

In [51]:
from tensorflow.keras import layers, models

# Definición de una capa de la red neuronal con los componentes personalizados
layer = keras.layers.Dense(1, activation=my_softplus,
                           kernel_initializer=my_glorot_initializer,
                           kernel_regularizer=my_l1_regularizer,
                           kernel_constraint=my_positive_weights)

In [52]:
network = models.Sequential()

network.add(layers.Dense(30, activation='relu', input_shape=X_train.shape[1:]))
network.add(layers.Dense(10, activation='relu'))
network.add(layers.Dense(1, activation=my_softplus,
                         kernel_initializer=my_glorot_initializer,
                         kernel_regularizer=my_l1_regularizer,
                         kernel_constraint=my_positive_weights))

En el caso de que la funcion contenga parametros que queremos almacenar/guardar junto al modelo, la mejor forma es crear una subclase de _tf.keras.*_

In [53]:
# Función de regularización personalizada
@tf.keras.utils.register_keras_serializable()
class MyL1Regularizer(tf.keras.regularizers.Regularizer):
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(self.factor * weights))
    def get_config(self):
        return {"factor": self.factor}

In [54]:
network = models.Sequential()

network.add(layers.Dense(30, activation='relu', input_shape=X_train.shape[1:]))
network.add(layers.Dense(10, activation='relu'))
network.add(layers.Dense(1, activation=my_softplus,
                         kernel_initializer=my_glorot_initializer,
                         kernel_regularizer=MyL1Regularizer(0.01),
                         kernel_constraint=my_positive_weights))

In [55]:
network.save("modelo_personalizado.keras")

In [57]:
model = keras.models.load_model(
    "modelo_personalizado.keras",
    custom_objects={
        "MyL1Regularizer": MyL1Regularizer,
        "my_positive_weights": my_positive_weights,
        "my_glorot_initializer": my_glorot_initializer,
        "my_softplus": my_softplus,
        "Custom>my_softplus": my_softplus,  # Añadimos este mapeo exacto
    },
)

# Metricas personalizadas

Se utilizan exactamente igual que todos los componentes vistos anteriormente

In [64]:
# Función que genera al función de pérdida de Huber
@tf.keras.utils.register_keras_serializable()
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) <= threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = threshold * tf.abs(error) - threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    return huber_fn

In [65]:
network.compile(loss="mse", optimizer="sgd", metrics=[create_huber(2.0)])